In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # MobileNet needs 224x224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, 
                                         download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, 
                                         download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False)

print("Training samples:", len(trainset))  
print("Test samples:", len(testset))       

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, 10)
model = model.to(device)
print("MobileNet loaded successfully!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))

class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.smoothing = smoothing 

    def forward(self, predictions, targets):
        num_classes = predictions.size(1)
        
        one_hot = torch.zeros_like(predictions).scatter(1, targets.unsqueeze(1), 1)
        
        smooth_labels = one_hot * (1 - self.smoothing) + self.smoothing / num_classes
        
        log_probs = torch.log_softmax(predictions, dim=1)
        loss = -(smooth_labels * log_probs).sum(dim=1).mean()
        return loss


class OutputPenaltyLoss(nn.Module):

    def __init__(self, penalty=0.01):
        super(OutputPenaltyLoss, self).__init__()
        self.penalty = penalty

    def forward(self, predictions, targets):

        ce_loss = nn.functional.cross_entropy(
            predictions,
            targets
        )

        output_penalty = torch.mean(
            torch.abs(predictions)
        )

        total_loss = ce_loss + self.penalty * output_penalty

        return total_loss

custom_loss = LabelSmoothingLoss(smoothing=0.1)
standard_loss = nn.CrossEntropyLoss()
third_loss = OutputPenaltyLoss(penalty=0.01)

print("Loss functions ready!!")

def train_model(loss_fn, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    train_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    return train_losses, train_accuracies

    print("----Training with Standard Cross Entropy Loss ----")
std_losses, std_accs = train_model(standard_loss, epochs=5)


model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, 10)
model = model.to(device)

print("=== Training with Custom Label Smoothing Loss ===")
custom_losses, custom_accs = train_model(custom_loss, epochs=5)



model = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

model.classifier[1] = nn.Linear(
    model.last_channel,
    10
)

model = model.to(device)

print("---Training with Output Penalty Loss ---")

third_losses, third_accs = train_model(
    third_loss,
    epochs=5
)

epochs = range(1, 6)

plt.figure(figsize=(12, 4))


plt.subplot(1, 2, 1)
plt.plot(epochs, std_losses, label='Standard Loss', marker='o', color='red')
plt.plot(epochs, custom_losses, label='Custom Label Smoothing', marker='o', color='blue')

plt.plot(
    epochs,
    third_losses,
    label='Output Penalty',
    marker='o'
)


plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)


plt.subplot(1, 2, 2)
plt.plot(epochs, std_accs, label='Standard Loss', marker='o', color='red')
plt.plot(epochs, custom_accs, label='Custom Label Smoothing', marker='o', color='blue')
plt.plot(
    epochs,
    third_accs,
    label='Output Penalty',
    marker='o'
)
plt.title('Training Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('results.png')
plt.show()

print("\n---- FINAL SUMMARY ----")
print(f"Standard Loss  - Final Accuracy: {std_accs[-1]:.2f}%")
print(f"Custom Loss    - Final Accuracy: {custom_accs[-1]:.2f}%")
print(f"Third Loss     - Final Accuracy: {third_accs[-1]:.2f}%")
print(f"Improvement    : +{custom_accs[-1] - std_accs[-1]:.2f}%")
print(f"Model Size     : 2,236,682 parameters (~2.2M)") 



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, 
                                         download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, 
                                         download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False)

print("Training samples:", len(trainset))  
print("Test samples:", len(testset))      

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, 10)
model = model.to(device)
print("MobileNet loaded successfully!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))

class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.smoothing = smoothing  

    def forward(self, predictions, targets):
        num_classes = predictions.size(1)
        
        one_hot = torch.zeros_like(predictions).scatter(1, targets.unsqueeze(1), 1)
        
        smooth_labels = one_hot * (1 - self.smoothing) + self.smoothing / num_classes
        
        log_probs = torch.log_softmax(predictions, dim=1)
        loss = -(smooth_labels * log_probs).sum(dim=1).mean()
        return loss


class OutputPenaltyLoss(nn.Module):

    def __init__(self, penalty=0.01):
        super(OutputPenaltyLoss, self).__init__()
        self.penalty = penalty

    def forward(self, predictions, targets):

        ce_loss = nn.functional.cross_entropy(
            predictions,
            targets
        )

        output_penalty = torch.mean(
            torch.abs(predictions)
        )

        total_loss = ce_loss + self.penalty * output_penalty

        return total_loss

custom_loss = LabelSmoothingLoss(smoothing=0.1)
standard_loss = nn.CrossEntropyLoss()
third_loss = OutputPenaltyLoss(penalty=0.01)

print("Loss functions ready!!")

def train_model(loss_fn, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    train_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    return train_losses, train_accuracies

    print("----Training with Standard Cross Entropy Loss ----")
std_losses, std_accs = train_model(standard_loss, epochs=5)#


model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, 10)
model = model.to(device)

print("=== Training with Custom Label Smoothing Loss ===")
custom_losses, custom_accs = train_model(custom_loss, epochs=5)#




model = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

model.classifier[1] = nn.Linear(
    model.last_channel,
    10
)

model = model.to(device)

print("---Training with Output Penalty Loss ---")

third_losses, third_accs = train_model(
    third_loss,
    epochs=5#
)

epochs = range(1, 6)

plt.figure(figsize=(12, 4))


plt.subplot(1, 2, 1)
plt.plot(epochs, std_losses, label='Standard Loss', marker='o', color='red')
plt.plot(epochs, custom_losses, label='Custom Label Smoothing', marker='o', color='blue')

plt.plot(
    epochs,
    third_losses,
    label='Output Penalty',
    marker='o'
)


plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

### Accuracy Graphs
plt.subplot(1, 2, 2)
plt.plot(epochs, std_accs, label='Standard Loss', marker='o', color='red')
plt.plot(epochs, custom_accs, label='Custom Label Smoothing', marker='o', color='blue')
plt.plot(
    epochs,
    third_accs,
    label='Output Penalty',
    marker='o'
)
plt.title('Training Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('results.png')
plt.show()

print("\n---- FINAL SUMMARY ----")
print(f"Standard Loss  - Final Accuracy: {std_accs[-1]:.2f}%")
print(f"Custom Loss    - Final Accuracy: {custom_accs[-1]:.2f}%")
print(f"Third Loss     - Final Accuracy: {third_accs[-1]:.2f}%")
print(f"Improvement    : +{custom_accs[-1] - std_accs[-1]:.2f}%")
print(f"Model Size     : 2,236,682 parameters (~2.2M)") 

